# 1、基于LCEL语法的新的chain:create_sql_query_chain

SQLDatabase的使用

In [2]:
from langchain.chains.sql_database.query import create_sql_query_chain
from langchain_community.utilities.sql_database import SQLDatabase

#访问mysql数据库
## mysql+pymysql://用户名:密码@ip地址:端口号/数据库名
db_name = "root"
db_password = "123"
db_host = "192.168.19.128"
db_port = 3306
db_database = "py11"
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_name}:{db_password}@{db_host}:{db_port}/{db_database}")

#查询对应的数据库中包含的表
# print(db.get_usable_table_names())


# db.run("查新employees表中有多少条记录？")  #错误的
db.run("SELECT COUNT(*) FROM employees")

'[(10,)]'

In [3]:
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 1、获取大模型
chat_model = ChatOpenAI(
    model="gpt-4o-mini"
)


chain = create_sql_query_chain(
    llm = chat_model,
    db = db,
)

# chain.invoke({"question":"查询employees表中有多少条记录？"})
chain.invoke({"question":"查询employees表中最高薪资的员工和其最高薪资"})

'SQLQuery: SELECT `name`, `salary` FROM `employees` ORDER BY `salary` DESC LIMIT 1'

# 2、LCEL语法之create_stuff_documents_chain （了解,不重要）



In [4]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

# 定义提示词模板
# prompt = PromptTemplate.from_template("""
# 如下文档{docs}中说，香蕉是什么颜色的？
# """)

prompt = PromptTemplate.from_template("""
如下文档{docs}的内容，回复{question}？
""")

# 创建链
llm = ChatOpenAI(model="gpt-4o-mini")
chain = create_stuff_documents_chain(llm, prompt, document_variable_name="docs")

# 文档输入
docs = [
    Document(
        page_content="苹果，学名Malus pumila Mill.，别称西洋苹果、柰，属于蔷薇科苹果属的植物。苹果是全球最广泛种植和销售的水果之一，具有悠久的栽培历史和广泛的分布范围。苹果的原始种群主要起源于中亚的天山山脉附近，尤其是现代哈萨克斯坦的阿拉木图地区，提供了所有现代苹果品种的基因库。苹果通过早期的贸易路线，如丝绸之路，从中亚向外扩散到全球各地。"
    ),
    Document(
        page_content="香蕉是白色的水果，主要产自热带地区。"

    ),
    Document(
        page_content="蓝莓是蓝色的浆果，含有抗氧化物质。"

    ),
    Document(
        page_content="我的名字叫小明。"

    )
]
# 执行摘要
chain.invoke({"docs": docs,"question":"我的名字叫什么？"})

'你的名字叫小明。'

代码的功能是：**将多个文档内容整合后，让大模型基于这些文档来回答问题**。

## 核心功能详解

### 1. **`create_stuff_documents_chain` 的作用**

这是 LangChain 提供的一个**文档问答链**，它的工作流程是：

```
多个文档 → 合并成一段文本 → 插入提示词 → 发送给 LLM → 返回答案
```

**"Stuff"** 的含义是 **"填充/塞入"**，即把所有文档内容**一次性塞入**提示词中。

---

## 2. **代码执行流程**

### 步骤1：准备文档
```python
docs = [
    Document(page_content="苹果..."),
    Document(page_content="香蕉是白色的水果..."),
    Document(page_content="蓝莓..."),
    Document(page_content="我的名字叫小明。")
]
```

### 步骤2：文档内容合并
`create_stuff_documents_chain` 会自动将所有文档的 `page_content` 合并成一段文本：

```
苹果，学名Malus pumila Mill.，别称西洋苹果...

香蕉是白色的水果，主要产自热带地区。

蓝莓是蓝色的浆果，含有抗氧化物质。

我的名字叫小明。
```

### 步骤3：填充到提示词模板
```python
prompt = PromptTemplate.from_template("""
如下文档{docs}的内容，回复{question}？
""")
```

实际发送给模型的提示词：
```
如下文档
苹果，学名Malus pumila Mill....
香蕉是白色的水果...
蓝莓是蓝色的浆果...
我的名字叫小明。

的内容，回复我的名字叫什么？？
```

### 步骤4：模型返回答案
```python
chain.invoke({"docs": docs, "question": "我的名字叫什么？"})
```

**预期输出**：
```
"根据文档内容，你的名字叫小明。"
```

---

## 3. **关键参数说明**

### `document_variable_name="docs"`
- 指定文档内容在提示词中的**变量名**
- 必须与 `PromptTemplate` 中的 `{docs}` 对应
- 如果不指定，默认是 `"context"`

### 示例对比

#### 使用 `docs`（当前代码）：
```python
prompt = PromptTemplate.from_template("如下文档{docs}的内容...")
chain = create_stuff_documents_chain(llm, prompt, document_variable_name="docs")
```

#### 使用默认的 `context`：
```python
prompt = PromptTemplate.from_template("如下文档{context}的内容...")
chain = create_stuff_documents_chain(llm, prompt)  # 默认就是 "context"
```

---

## 4. **适用场景**

这种方法适合：
- ✅ **文档数量少**（3-5个）
- ✅ **文档内容短**（每个几百字以内）
- ✅ **需要综合多个文档的信息**来回答问题

### 不适合的场景
- ❌ 文档太多（超过10个）
- ❌ 文档太长（总字数超过模型上下文限制）
- ❌ 需要精确检索（此时应该用 RAG 向量检索）

---

## 5. **与 RAG 的区别**

| 特性 | Stuff Documents Chain | RAG（检索增强生成） |
|------|----------------------|-------------------|
| 文档处理 | 全部文档一次性输入 | 先检索相关文档再输入 |
| 适用文档量 | 少（3-5个） | 多（成百上千） |
| 成本 | 低（调用1次模型） | 较高（需要向量数据库） |
| 准确性 | 依赖模型从大段文本中提取 | 只输入相关片段，更精准 |

---

## 6. **完整示例输出**

运行这段代码后的完整对话：

**输入**：
```python
chain.invoke({"docs": docs, "question": "我的名字叫什么？"})
```

**模型实际看到的内容**：
```
如下文档
苹果，学名Malus pumila Mill.，别称西洋苹果、柰...
香蕉是白色的水果，主要产自热带地区。
蓝莓是蓝色的浆果，含有抗氧化物质。
我的名字叫小明。

的内容，回复我的名字叫什么？？
```

**返回结果**：
```python
{
    'output': '根据文档内容，你的名字叫小明。'
}
```

---

## 总结

这段代码的核心功能是：

1. **把多个文档整合成一段文本**
2. **和用户的问题一起发送给大模型**
3. **让模型基于文档内容回答问题**

这是一种简单的**基于文档的问答系统**，适合处理少量短文档的场景。如果文档很多或很长，应该考虑使用 RAG（检索增强生成）方案。